# Resume Screening & Domain Classification using NLP & Machine Learning

This project implements an automated Resume Screening and Classification pipeline. Given resume text, the model preprocesses the text, extracts TF-IDF n-gram features, and classifies the resume into one of **24 job categories** (e.g., Information Technology, Engineering, HR, Finance, Healthcare, etc.).

### Pipeline Overview:
1. **Exploratory Data Analysis (EDA):** Inspect distribution across 24 categories.
2. **Text Cleaning & Normalization:** Regex cleaning for URLs, mentions, hashtags, symbols, numbers, and whitespaces.
3. **Feature Extraction:** TF-IDF Vectorization with sublinear term-frequency scaling.
4. **Model Training & Comparison:**
   - Baseline: Multinomial Naive Bayes
   - Optimized: Calibrated Linear Support Vector Classifier (LinearSVC)
5. **Evaluation:** Accuracy, Classification Report, and Confusion Matrix.
6. **Inference on Unseen Resumes:** Test with real-world candidate profiles.

In [1]:
import re
import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set style
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Dataset

In [2]:
df = pd.read_csv('../data/Resume.csv')
print('Dataset shape:', df.shape)
df.head()

## 2. Dataset Information & Category Distribution

In [3]:
df.info()
print('\nCategory counts:\n', df['Category'].value_counts())

In [4]:
plt.figure(figsize=(12, 7))
sns.countplot(y=df['Category'], order=df['Category'].value_counts().index, palette='viridis')
plt.title('Resume Category Distribution (24 Categories)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Resumes')
plt.ylabel('Category')
plt.show()

## 3. Text Preprocessing & Cleaning
We clean raw resume text using regular expressions to remove unnecessary noise such as URLs, email/social handles, punctuation, digits, and extra whitespace.

In [5]:
def clean_resume(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+\s*', ' ', text)                  # remove URLs
    text = re.sub(r'\b(RT|cc)\b', ' ', text)                # remove RT and cc
    text = re.sub(r'#\S+', '', text)                         # remove hashtags
    text = re.sub(r'@\S+', ' ', text)                        # remove mentions
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)  # remove punctuations
    text = re.sub(r'[0-9]+', ' ', text)                      # remove numbers
    text = re.sub(r'[^\x00-\x7f]', ' ', text)                # remove non-ASCII chars
    text = re.sub(r'\s+', ' ', text)                         # normalize whitespace
    return text.strip()

df['Cleaned_Resume'] = df['Resume_str'].apply(clean_resume)
df[['Resume_str', 'Cleaned_Resume']].head()

## 4. Target Encoding & Train-Test Split

In [6]:
X = df['Cleaned_Resume']
y = df['Category']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')

## 5. TF-IDF Feature Extraction

In [7]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print('TF-IDF Train matrix shape:', X_train_tfidf.shape)

## 6. Model Training & Evaluation
### Model A: Multinomial Naive Bayes (Baseline)

In [8]:
model_nb = MultinomialNB(alpha=0.1)
model_nb.fit(X_train_tfidf, y_train)

y_pred_nb = model_nb.predict(X_test_tfidf)
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'Multinomial Naive Bayes Accuracy: {acc_nb:.4f} ({acc_nb*100:.2f}%)')
print('\nClassification Report (Naive Bayes):')
print(classification_report(y_test, y_pred_nb, target_names=le.classes_, zero_division=0))

### Model B: Calibrated LinearSVC (Enhanced Classifier)

In [9]:
base_svc = LinearSVC(C=1.0, dual='auto', random_state=42)
model_svc = CalibratedClassifierCV(base_svc, cv=3)
model_svc.fit(X_train_tfidf, y_train)

y_pred_svc = model_svc.predict(X_test_tfidf)
acc_svc = accuracy_score(y_test, y_pred_svc)
print(f'Calibrated LinearSVC Accuracy: {acc_svc:.4f} ({acc_svc*100:.2f}%)')
print('\nClassification Report (LinearSVC):')
print(classification_report(y_test, y_pred_svc, target_names=le.classes_, zero_division=0))

## 7. Confusion Matrix Visualization

In [10]:
cm = confusion_matrix(y_test, y_pred_svc)
plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix - LinearSVC ({acc_svc*100:.1f}% Accuracy)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 8. Test on Sample Unseen Resumes

In [11]:
sample_resumes = [
    """Data scientist skilled in Python, machine learning, deep learning, data analysis, pandas, numpy and predictive modeling.""",
    """Front-end web developer experienced in HTML, CSS, JavaScript, React, Bootstrap and responsive web design.""",
    """DevOps engineer experienced in Docker, Kubernetes, CI/CD pipelines, Jenkins and cloud deployment.""",
    """Software testing engineer skilled in manual testing, automation testing using Selenium and bug tracking tools.""",
    """Graphic designer experienced in Adobe Photoshop, Illustrator, UI/UX design, branding and digital media design.""",
    """High school mathematics teacher experienced in curriculum planning, student assessment, classroom management and interactive teaching.""",
    """Network security specialist experienced in cybersecurity, firewalls, intrusion detection systems and network monitoring."""
]

for resume in sample_resumes:
    cleaned = clean_resume(resume)
    vec = tfidf.transform([cleaned])
    pred_idx = model_svc.predict(vec)[0]
    proba = np.max(model_svc.predict_proba(vec)[0])
    pred_cat = le.inverse_transform([pred_idx])[0]
    
    print(f'Resume: {resume[:75]}...')
    print(f'Predicted Domain: {pred_cat} (Confidence: {proba*100:.1f}%)')
    print('-' * 60)